# Affinity Classification Diagnostics

This notebook computes ordinal affinity diagnostics on `(abstract, question)` pairs.

Metrics included:
1. Per-model class metrics
2. Per-model quadratic weighted kappa
3. Per-model mean signed error
4. An additional synthetic model formed by averaging predictions across models

In [73]:
from pathlib import Path

import importlib
import itertools
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import GroupKFold
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [74]:
# Label definitions (edit only if your taxonomy changes).
LABEL_ORDER = ["Low", "Moderate", "High"]
LABEL_TO_NUM = {"Low": 0, "Moderate": 1, "High": 2}
NUM_TO_LABEL = {0: "Low", 1: "Moderate", 2: "High"}


def normalize_label(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, np.integer, float, np.floating)):
        x = float(v)
        # Numeric predictions in merged files are raw affinity scores (0-120).
        if 0 <= x < 40:
            return "Low"
        if 40 <= x < 80:
            return "Moderate"
        if x >= 80:
            return "High"
        return np.nan
    text = str(v).strip()
    text_low = text.lower()
    if text_low in {"low", "moderate", "high"}:
        return text_low.capitalize()
    return np.nan


# Hardcoded column names used throughout this notebook.
expert_col = "Evaluator_Affinity_Level"
prediction_col = "LLM_Affinity"
abstract_id_col = "Abstract_Index"
model_col = "Model_Slug"

## Load Data

The loader tries likely project files first (including benchmark and calibration outputs), then you can override `input_path` manually if needed.

In [86]:
repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bench_root = repo_root / "data" / "results" / "affinity_benchmark"
calib_root = repo_root / "data" / "processed" / "affinity_calibration"
benchmark_input_filename = "merged_ra_affinities.csv"
calibration_input_filename = "calibration_abstract_question_affinities.csv"

select = 'calibration'
# select = 'benchmark'
if select == 'calibration':
    benchmark_run_dir_names = ["20260724_155639"]
    calibration_run_dir_names = ["20260618_Mark_batch1_v4"]
elif select == 'benchmark':
    benchmark_run_dir_names = ["20260726_134607_5"]
    calibration_run_dir_names = ["20260728_abstracts_evaluation_template_mark_om_v1"]
def load_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin1"):
            try:
                return pd.read_csv(path, encoding=encoding)
            except UnicodeDecodeError:
                continue
        raise UnicodeDecodeError(
            "utf-8", b"", 0, 1, f"Unable to decode CSV file: {path}"
        )
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file extension for {path}")

def collect_frames(root: Path, run_dir_names, input_filename: str, source_name: str):
    if isinstance(run_dir_names, (str, Path)):
        run_dir_names = [run_dir_names]

    run_dirs = sorted([p for p in root.iterdir() if p.is_dir()])
    if not run_dirs:
        raise FileNotFoundError(f"No run directories found under {root}")

    selected_run_dirs = [p for p in run_dirs if p.name in {str(x) for x in run_dir_names}]
    if not selected_run_dirs:
        raise FileNotFoundError(f"No matching run directories found for: {run_dir_names}")

    frames = []
    for run_dir in selected_run_dirs:
        input_path = run_dir / input_filename
        if not input_path.exists():
            raise FileNotFoundError(f"Missing file: {input_path}")

        run_df = load_table(input_path)
        run_df["Run_ID"] = run_dir.name
        run_df["Source_Type"] = source_name
        frames.append(run_df)

    return frames, selected_run_dirs

benchmark_frames, selected_benchmark_dirs = collect_frames(
    bench_root, benchmark_run_dir_names, benchmark_input_filename, "benchmark"
)
calibration_frames, selected_calibration_dirs = collect_frames(
    calib_root, calibration_run_dir_names, calibration_input_filename, "calibration"
)

bench_df = pd.concat(benchmark_frames, ignore_index=True)
calib_df = pd.concat(calibration_frames, ignore_index=True)
df_raw = pd.merge(
    bench_df,
    calib_df,
    on=["Abstract_Index", "RA2025_ID"],
    how="inner",
    suffixes=("_bench", "_calib"),
)

print("Benchmark run directories:", [p.name for p in selected_benchmark_dirs])
print("Calibration run directories:", [p.name for p in selected_calibration_dirs])
print("Calibration abstracts:", calib_df['Abstract_Index'].nunique())
print("Benchmark abstracts:", bench_df['Abstract_Index'].nunique())
print("Rows loaded:", len(df_raw))
print(f"Columns ({len(df_raw.columns)}): {list(df_raw.columns)}")
display(df_raw.head(10))

Benchmark run directories: ['20260724_155639']
Calibration run directories: ['20260618_Mark_batch1_v4']
Calibration abstracts: 21
Benchmark abstracts: 21
Rows loaded: 650
Columns (19): ['Abstract_Index', 'Document Title_bench', 'RA2025_ID', 'RA_Question_bench', 'Cosine_x100', 'LLM_Affinity', 'LLM_Affinity_Reason', 'Model_Slug', 'Run_ID_bench', 'Source_Type_bench', 'Abstract_Index_Old', 'Document Title_calib', 'RA_Question_calib', 'Evaluator_Affinity', 'Evaluator_Affinity_Level', 'Evaluator_Affinity_Reason', 'Source_Evaluator', 'Run_ID_calib', 'Source_Type_calib']


,Abstract_Index,Document Title_bench,RA2025_ID,RA_Question_bench,Cosine_x100,LLM_Affinity,LLM_Affinity_Reason,Model_Slug,Run_ID_bench,Source_Type_bench,Abstract_Index_Old,Document Title_calib,RA_Question_calib,Evaluator_Affinity,Evaluator_Affinity_Level,Evaluator_Affinity_Reason,Source_Evaluator,Run_ID_calib,Source_Type_calib
0,10.1109/TPWRS.2023.3234277,Scenario Reduction Network Based on Wasserstei...,41,what additional probabilistic planning methods...,48.1018,90.0,The abstract directly addresses scenario reduc...,gemini-3.1-flash-lite,20260724_155639,benchmark,1,Scenario Reduction Network Based on Wasserstei...,What additional probabilistic planning methods...,70,High,NaN,Mark,20260618_Mark_batch1_v4,calibration
1,10.1109/TPWRS.2023.3234277,Scenario Reduction Network Based on Wasserstei...,42,what additional planning models and methods ar...,41.1134,85.0,The abstract provides a direct methodological ...,gemini-3.1-flash-lite,20260724_155639,benchmark,1,Scenario Reduction Network Based on Wasserstei...,What additional planning models and methods ar...,50,Moderate,NaN,Mark,20260618_Mark_batch1_v4,calibration
2,10.1109/TPWRS.2023.3234277,Scenario Reduction Network Based on Wasserstei...,47,how do system operators adequately account for...,40.4033,50.0,Scenario reduction is a key technique for stud...,gemini-3.1-flash-lite,20260724_155639,benchmark,1,Scenario Reduction Network Based on Wasserstei...,How do system operators adequately account for...,30,Low,NaN,Mark,20260618_Mark_batch1_v4,calibration
3,10.1109/TPWRS.2023.3234277,Scenario Reduction Network Based on Wasserstei...,35,"what models and methods, including testing, ar...",34.7982,40.0,The abstract provides tools for handling uncer...,gemini-3.1-flash-lite,20260724_155639,benchmark,1,Scenario Reduction Network Based on Wasserstei...,"What models and methods, including testing, ar...",30,Low,NaN,Mark,20260618_Mark_batch1_v4,calibration
4,10.1109/TPWRS.2023.3234277,Scenario Reduction Network Based on Wasserstei...,48,what additional planning models and methods ar...,34.3618,70.0,Scenario reduction is critical for modeling lo...,gemini-3.1-flash-lite,20260724_155639,benchmark,1,Scenario Reduction Network Based on Wasserstei...,What additional planning models and methods ar...,50,Moderate,NaN,Mark,20260618_Mark_batch1_v4,calibration
5,10.1109/TPWRS.2023.3234277,Scenario Reduction Network Based on Wasserstei...,43,what changes can be incorporated into the tran...,33.7334,50.0,Improved scenario reduction techniques can enh...,gemini-3.1-flash-lite,20260724_155639,benchmark,1,Scenario Reduction Network Based on Wasserstei...,What changes can be incorporated into the tran...,50,Moderate,NaN,Mark,20260618_Mark_batch1_v4,calibration
6,10.1109/TPWRS.2023.3234277,Scenario Reduction Network Based on Wasserstei...,24,how can the risk of high-impact low-probabilit...,32.3158,40.0,Scenario reduction is relevant to risk assessm...,gemini-3.1-flash-lite,20260724_155639,benchmark,1,Scenario Reduction Network Based on Wasserstei...,How can the risk of high-impact low-probabilit...,50,Moderate,NaN,Mark,20260618_Mark_batch1_v4,calibration
7,10.1109/TPWRS.2023.3234277,Scenario Reduction Network Based on Wasserstei...,39,what technical models need to be added to long...,29.4058,40.0,Scenario reduction is a general planning tool ...,gemini-3.1-flash-lite,20260724_155639,benchmark,1,Scenario Reduction Network Based on Wasserstei...,What technical models need to be added to long...,30,Low,NaN,Mark,20260618_Mark_batch1_v4,calibration
8,10.1109/TPWRS.2023.3234277,Scenario Reduction Network Based on Wasserstei...,37,what studies and metrics are required to ident...,25.6859,50.0,Scenario reduction is a key method for identif...,gemini-3.1-flash-lite,20260724_155639,benchmark,1,Scenario Reduction Network Based on Wasserstei...,What studies and metrics are required to ident...,50,Moderate,NaN,Mark,20260618_Mark_batch1_v4,calibration
9,10.1109/TPWRS.2023.3234277,Scenario Reduction Network Based on Wass

In [87]:
required_cols = [expert_col, prediction_col, abstract_id_col]
missing_required = [c for c in required_cols if c not in df_raw.columns]

print("Hardcoded column mapping in use:")
print(f"  expert_col      -> {expert_col}")
print(f"  prediction_col  -> {prediction_col}")
print(f"  abstract_id_col -> {abstract_id_col}")

if missing_required:
    raise ValueError(
        f"Missing required columns in df_raw: {missing_required}. "
        "Please update hardcoded column names near the top of this notebook."
    )

Hardcoded column mapping in use:
  expert_col      -> Evaluator_Affinity_Level
  prediction_col  -> LLM_Affinity
  abstract_id_col -> Abstract_Index


## Clean Labels

The cleaner normalizes variants such as `low`, `LOW`, `med`, `medium`, and `high`, then maps labels to ordinal values:

- Low = 0
- Medium = 1
- High = 2

In [88]:
# Assume labels are canonical: 'Low', 'Moderate', 'High'.
eval_df = df_raw.copy()
rows_before = len(eval_df)

expert_labels = eval_df[expert_col].map(normalize_label)
pred_labels = eval_df[prediction_col].map(normalize_label)
valid_mask = expert_labels.notna() & pred_labels.notna()

eval_df = eval_df.loc[valid_mask].copy()
eval_df['expert_num'] = expert_labels.loc[valid_mask].map(LABEL_TO_NUM).astype(int).to_numpy()
eval_df['pred_num'] = pred_labels.loc[valid_mask].map(LABEL_TO_NUM).astype(int).to_numpy()

rows_after = len(eval_df)
rows_removed = rows_before - rows_after

print(f'Rows before cleaning: {rows_before:,}')
print(f'Rows after cleaning:  {rows_after:,}')
print(f'Rows removed:         {rows_removed:,}')

print(f'Evaluated (abstract, question) pairs after cleaning: {rows_after:,}')
if abstract_id_col in eval_df.columns:
    print(f'Unique abstracts: {eval_df[abstract_id_col].nunique():,}')
else:
    print('Unique abstracts: not available (abstract_id_col not found).')
print(f'Unique abstract indices: {df_raw.Abstract_Index.unique().size:,}')

Rows before cleaning: 650
Rows after cleaning:  514
Rows removed:         136
Evaluated (abstract, question) pairs after cleaning: 514
Unique abstracts: 21
Unique abstract indices: 21


## Model-wise Metrics

In [89]:
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

affinity_metrics = importlib.import_module("src.analysis.affinity_metrics")
AVERAGE_MODEL_NAME = "Average across models"
BIAS_CORRECTED_AVERAGE_MODEL_NAME = "Average across models (bias-corrected)"


def build_average_model_frame(df: pd.DataFrame) -> pd.DataFrame:
    # Average raw 0-120 scores across models first, then discretise (matches the
    # project's Affinity_Mean convention), rather than averaging discrete levels.
    averaged_df = (
        df.groupby([abstract_id_col, "RA2025_ID"], as_index=False)
        .agg(
            expert_num=("expert_num", "first"),
            expert_raw=("Evaluator_Affinity", "first"),
            raw_score_mean=(prediction_col, "mean"),
        )
    )
    averaged_df["pred_num"] = (
        averaged_df["raw_score_mean"].map(normalize_label).map(LABEL_TO_NUM).astype(int)
    )
    return averaged_df


def build_bias_corrected_average_model_frame(
    averaged_df: pd.DataFrame,
) -> tuple[pd.DataFrame, float]:
    score_bias = float((averaged_df["raw_score_mean"] - averaged_df["expert_raw"]).mean())
    corrected_df = averaged_df.copy()
    corrected_df["pred_num"] = (
        corrected_df["raw_score_mean"]
        .sub(score_bias)
        .clip(0, 120)
        .map(normalize_label)
        .map(LABEL_TO_NUM)
        .astype(int)
    )
    return corrected_df, score_bias


def build_metrics_for_model(df: pd.DataFrame, model_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    y_true = df["expert_num"].to_numpy()
    y_pred = df["pred_num"].to_numpy()

    class_metrics_df = affinity_metrics.build_affinity_class_metrics(
        y_true,
        y_pred,
        labels=list(LABEL_TO_NUM.values()),
        class_names=LABEL_ORDER,
    ).copy()
    class_metrics_df.insert(0, "Model", model_name)

    overall_metrics_df = pd.DataFrame(
        [
            {
                "Model": model_name,
                "Quadratic Weighted Kappa": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
                "Mean signed error": float(np.mean(y_pred - y_true)),
                "Pairs": int(len(df)),
                "Unique abstracts": int(df[abstract_id_col].nunique()),
            }
        ]
    )
    return class_metrics_df, overall_metrics_df

In [90]:
model_class_metrics_frames = []
model_overall_metrics_frames = []

for model_name, model_df in eval_df.groupby(model_col, sort=True):
    class_metrics_df, overall_metrics_df = build_metrics_for_model(model_df, model_name)
    model_class_metrics_frames.append(class_metrics_df)
    model_overall_metrics_frames.append(overall_metrics_df)

average_model_df = build_average_model_frame(eval_df)
average_class_metrics_df, average_overall_metrics_df = build_metrics_for_model(
    average_model_df,
    AVERAGE_MODEL_NAME,
)
bias_corrected_average_model_df, ensemble_score_bias = build_bias_corrected_average_model_frame(
    average_model_df
)
bias_corrected_class_metrics_df, bias_corrected_overall_metrics_df = build_metrics_for_model(
    bias_corrected_average_model_df,
    BIAS_CORRECTED_AVERAGE_MODEL_NAME,
)

class_metrics_df = pd.concat(
    model_class_metrics_frames + [average_class_metrics_df, bias_corrected_class_metrics_df],
    ignore_index=True,
)
overall_metrics_df = pd.concat(
    model_overall_metrics_frames + [average_overall_metrics_df, bias_corrected_overall_metrics_df],
    ignore_index=True,
)

class_metrics_df["Class"] = pd.Categorical(class_metrics_df["Class"], categories=LABEL_ORDER, ordered=True)
class_metrics_df = class_metrics_df.sort_values(["Model", "Class"]).reset_index(drop=True)
overall_metrics_df = overall_metrics_df.sort_values("Model").reset_index(drop=True)

print(f"Estimated ensemble raw-score bias: {ensemble_score_bias:+.2f}")
display(overall_metrics_df)
display(class_metrics_df[["Model", "Class", "Support", "Precision", "Recall", "One_vs_Rest_Accuracy"]])

Estimated ensemble raw-score bias: +6.15


,Model,Quadratic Weighted Kappa,Mean signed error,Pairs,Unique abstracts
0,Average across models,0.562996,0.015385,130,21
1,Average across models (bias-corrected),0.481998,-0.138462,130,21
2,gemini-3.1-flash-lite,0.507937,0.282258,124,20
3,gemma4-31b-cloud,0.425809,0.246154,130,21
4,gpt-5.4-mini,0.524370,-0.123077,130,21
5,mistral-large-3-675b-cloud,0.465795,0.161538,130,21


,Model,Class,Support,Precision,Recall,One_vs_Rest_Accuracy
0,Average across models,Low,67,0.730159,0.686567,0.707692
1,Average across models,Moderate,49,0.563636,0.632653,0.676923
2,Average across models,High,14,0.750000,0.642857,0.938462
3,Average across models (bias-corrected),Low,67,0.666667,0.776119,0.684615
4,Average across models (bias-corrected),Moderate,49,0.511111,0.469388,0.630769
5,Average across models (bias-corrected),High,14,0.714286,0.357143,0.915385
6,gemini-3.1-flash-lite,Low,64,0.823529,0.437500,0.661290
7,gemini-3.1-flash-lite,Moderate,47,0.486111,0.744681,0.604839
8,gemini-3.1-flash-lite,High,13,0.555556,0.769231,0.911290
9,gemma4-31b-cloud,Low,67,0.720930,0.462687,0.630769


In [91]:
class_metrics_df[class_metrics_df["Class"] == "Moderate"][["Model", "Support", "Precision", "Recall", "One_vs_Rest_Accuracy"]]

,Model,Support,Precision,Recall,One_vs_Rest_Accuracy
1,Average across models,49,0.563636,0.632653,0.676923
4,Average across models (bias-corrected),49,0.511111,0.469388,0.630769
7,gemini-3.1-flash-lite,47,0.486111,0.744681,0.604839
10,gemma4-31b-cloud,49,0.461538,0.612245,0.584615
13,gpt-5.4-mini,49,0.621622,0.469388,0.692308
16,mistral-large-3-675b-cloud,49,0.519231,0.551020,0.638462


In [92]:
# Compare each model against the "Average across models" baseline, per class.
# Support is excluded here: it's a sample count, not a higher-is-better metric,
# so "better/worse than average" doesn't apply to it.
delta_metrics = ["Precision", "Recall", "One_vs_Rest_Accuracy"]
colors = {"Low": "#636EFA", "Moderate": "#EF553B", "High": "#00CC96"}

average_metrics_df = (
    class_metrics_df.loc[class_metrics_df["Model"] == AVERAGE_MODEL_NAME, ["Class"] + delta_metrics]
    .rename(columns={metric: f"{metric}_avg" for metric in delta_metrics})
)

delta_df = class_metrics_df[class_metrics_df["Model"] != AVERAGE_MODEL_NAME].merge(
    average_metrics_df, on="Class", how="left"
)
for metric in delta_metrics:
    delta_df[f"{metric}_delta"] = delta_df[metric] - delta_df[f"{metric}_avg"]

model_order = sorted(delta_df["Model"].unique())

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"{m} (Δ vs average)" for m in delta_metrics],
    horizontal_spacing=0.08,
)

for col_idx, metric in enumerate(delta_metrics, start=1):
    for class_label in LABEL_ORDER:
        class_data = (
            delta_df[delta_df["Class"] == class_label]
            .set_index("Model")
            .reindex(model_order)
            .reset_index()
        )
        fig.add_trace(
            go.Bar(
                x=class_data["Model"],
                y=class_data[f"{metric}_delta"],
                name=class_label,
                marker=dict(color=colors[class_label], line=dict(color="#1f1f1f", width=0.5)),
                legendgroup=class_label,
                showlegend=(col_idx == 1),
                customdata=np.stack([class_data[metric], class_data[f"{metric}_avg"]], axis=-1),
                hovertemplate=(
                    f"<b>{class_label}</b><br>"
                    "Model: %{x}<br>"
                    f"{metric}: %{{customdata[0]:.3f}}<br>"
                    "Average: %{customdata[1]:.3f}<br>"
                    "Δ: %{y:+.3f}"
                    "<extra></extra>"
                ),
            ),
            row=1,
            col=col_idx,
        )
    fig.add_hline(
        y=0,
        line_width=1,
        line_color="#1f1f1f",
        annotation_text="Average",
        annotation_position="top left",
        row=1,
        col=col_idx,
    )
    fig.update_yaxes(title_text="Δ vs average", row=1, col=col_idx)
    fig.update_xaxes(title_text="Model", row=1, col=col_idx)

fig.update_layout(
    title_text="Model Performance Relative to Cross-Model Average",
    barmode="group",
    bargap=0.2,
    bargroupgap=0.05,
    height=500,
    hovermode="closest",
)

fig.show()


## Robustness of the Model Ranking

Quadratic weighted kappa is estimated from 601 pairs, so small gaps in the table above may be
noise. Resampling pairs with replacement gives an interval around each model's QWK and a paired
comparison between the ensemble and the strongest individual model.


In [93]:
BOOTSTRAP_DRAWS = 1000
BOOTSTRAP_SEED = 0

QWK_WEIGHTS = (np.subtract.outer(np.arange(3), np.arange(3)) ** 2) / 4.0


def quadratic_weighted_kappa(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Same value as cohen_kappa_score(..., weights="quadratic"), fast enough to bootstrap."""
    observed = np.zeros((3, 3))
    np.add.at(observed, (y_true, y_pred), 1)
    expected = np.outer(observed.sum(axis=1), observed.sum(axis=0)) / observed.sum()
    return 1.0 - (QWK_WEIGHTS * observed).sum() / (QWK_WEIGHTS * expected).sum()


pair_index = [abstract_id_col, "RA2025_ID"]
level_matrix = pd.DataFrame(
    {name: g.set_index(pair_index)["pred_num"] for name, g in eval_df.groupby(model_col, sort=True)}
).dropna().astype(int)
level_matrix[AVERAGE_MODEL_NAME] = average_model_df.set_index(pair_index)["pred_num"]
expert_num_by_pair = average_model_df.set_index(pair_index).loc[level_matrix.index, "expert_num"].to_numpy()

rng = np.random.default_rng(BOOTSTRAP_SEED)
draw_rows = rng.integers(0, len(level_matrix), size=(BOOTSTRAP_DRAWS, len(level_matrix)))
bootstrap_kappa = pd.DataFrame(
    {
        name: [
            quadratic_weighted_kappa(expert_num_by_pair[rows], level_matrix[name].to_numpy()[rows])
            for rows in draw_rows
        ]
        for name in level_matrix.columns
    }
)

qwk_ci_df = (
    pd.DataFrame(
        {
            "Model": level_matrix.columns,
            "QWK": [
                quadratic_weighted_kappa(expert_num_by_pair, level_matrix[name].to_numpy())
                for name in level_matrix.columns
            ],
            "CI lower (2.5%)": bootstrap_kappa.quantile(0.025).to_numpy(),
            "CI upper (97.5%)": bootstrap_kappa.quantile(0.975).to_numpy(),
        }
    )
    .sort_values("QWK", ascending=False)
    .reset_index(drop=True)
)

best_single_model = qwk_ci_df.loc[qwk_ci_df["Model"] != AVERAGE_MODEL_NAME, "Model"].iloc[0]
ensemble_gain = bootstrap_kappa[AVERAGE_MODEL_NAME] - bootstrap_kappa[best_single_model]

display(qwk_ci_df)
print(f"Ensemble minus best single model ({best_single_model}):")
print(f"  point estimate      {ensemble_gain.mean():+.3f}")
print(f"  95% interval        [{ensemble_gain.quantile(0.025):+.3f}, {ensemble_gain.quantile(0.975):+.3f}]")
print(f"  P(ensemble better)  {(ensemble_gain > 0).mean():.2f}")

,Model,QWK,CI lower (2.5%),CI upper (97.5%)
0,Average across models,0.552577,0.383471,0.672809
1,gpt-5.4-mini,0.512021,0.304440,0.663923
2,gemini-3.1-flash-lite,0.507937,0.367337,0.633843
3,mistral-large-3-675b-cloud,0.470817,0.295426,0.607238
4,gemma4-31b-cloud,0.421087,0.264013,0.560911


Ensemble minus best single model (gpt-5.4-mini):
  point estimate      +0.039
  95% interval        [-0.071, +0.168]
  P(ensemble better)  0.73


## Discrimination vs. Score Scale

Two different failure modes are hidden inside a single QWK number: whether a model *orders*
`(abstract, question)` pairs the way the expert does, and whether it places those pairs on the
0-120 axis where the fixed 40/80 cut points expect them. Spearman rho is computed on raw scores,
so it measures ordering only and is unaffected by the cut points.


In [94]:
raw_score_matrix = eval_df.pivot_table(
    index=pair_index, columns=model_col, values=prediction_col, aggfunc="first"
).dropna()
raw_score_matrix[AVERAGE_MODEL_NAME] = raw_score_matrix.mean(axis=1)
expert_raw_by_pair = (
    eval_df.drop_duplicates(pair_index)
    .set_index(pair_index)
    .loc[raw_score_matrix.index, "Evaluator_Affinity"]
)

scale_df = (
    pd.DataFrame(
        {
            "Model": raw_score_matrix.columns,
            "Spearman rho (raw score vs expert)": [
                spearmanr(raw_score_matrix[name], expert_raw_by_pair).statistic
                for name in raw_score_matrix.columns
            ],
            "Mean raw score": raw_score_matrix.mean().to_numpy(),
            "Median raw score": raw_score_matrix.median().to_numpy(),
            "Share predicted >= 40": (raw_score_matrix >= 40).mean().to_numpy(),
        }
    )
    .sort_values("Spearman rho (raw score vs expert)", ascending=False)
    .reset_index(drop=True)
)

display(scale_df)
print(f"Expert mean raw affinity: {expert_raw_by_pair.mean():.1f}")
print(f"Expert share >= 40:       {(expert_raw_by_pair >= 40).mean():.3f}")

print("\nPairwise agreement on discrete levels (fraction of pairs given the same level):")
model_slugs = [name for name in raw_score_matrix.columns if name != AVERAGE_MODEL_NAME]
display(
    pd.DataFrame(
        {a: {b: (level_matrix[a] == level_matrix[b]).mean() for b in model_slugs} for a in model_slugs}
    )
)

,Model,Spearman rho (raw score vs expert),Mean raw score,Median raw score,Share predicted >= 40
0,Average across models,0.503697,45.530242,40.375,0.516129
1,mistral-large-3-675b-cloud,0.468890,47.379032,40.000,0.564516
2,gemma4-31b-cloud,0.460835,46.814516,40.000,0.669355
3,gemini-3.1-flash-lite,0.457111,52.056452,50.000,0.725806
4,gpt-5.4-mini,0.430943,35.870968,22.000,0.370968


Expert mean raw affinity: 39.0
Expert share >= 40:       0.484

Pairwise agreement on discrete levels (fraction of pairs given the same level):


,gemini-3.1-flash-lite,gemma4-31b-cloud,gpt-5.4-mini,mistral-large-3-675b-cloud
gemini-3.1-flash-lite,1.000000,0.685484,0.524194,0.637097
gemma4-31b-cloud,0.685484,1.000000,0.540323,0.637097
gpt-5.4-mini,0.524194,0.540323,1.000000,0.693548
mistral-large-3-675b-cloud,0.637097,0.637097,0.693548,1.000000


## Ensemble Aggregation: Raw Scores vs. Discretised Levels

`build_average_model_frame` averages the raw 0-120 scores first and then discretises the mean,
matching the project's `Affinity_Mean` convention. The variants below compare that choice against
averaging (or voting on) the already-discretised levels instead.



In [95]:
def discretize(scores: np.ndarray, lower_cut: float, upper_cut: float) -> np.ndarray:
    return np.where(scores < lower_cut, 0, np.where(scores < upper_cut, 1, 2))


aggregation_variants = {
    "Mean of raw scores, then discretise (current)": discretize(
        raw_score_matrix[AVERAGE_MODEL_NAME].to_numpy(), 40, 80
    ),
    "Mean of discrete levels": np.floor(level_matrix[model_slugs].mean(axis=1) + 0.5)
    .clip(0, 2)
    .astype(int)
    .to_numpy(),
    "Median of raw scores, then discretise": discretize(
        raw_score_matrix[model_slugs].median(axis=1).to_numpy(), 40, 80
    ),
    "Majority vote of discrete levels": level_matrix[model_slugs].mode(axis=1)[0].astype(int).to_numpy(),
}

aggregation_df = pd.DataFrame(
    [
        {
            "Aggregation": name,
            "Quadratic Weighted Kappa": quadratic_weighted_kappa(expert_num_by_pair, y_pred),
            "Mean signed error": float(np.mean(y_pred - expert_num_by_pair)),
            "Exact accuracy": float(np.mean(y_pred == expert_num_by_pair)),
        }
        for name, y_pred in aggregation_variants.items()
    ]
)

display(aggregation_df)

,Aggregation,Quadratic Weighted Kappa,Mean signed error,Exact accuracy
0,"Mean of raw scores, then discretise (current)",0.552577,0.024194,0.653226
1,Mean of discrete levels,0.528549,0.233871,0.637097
2,"Median of raw scores, then discretise",0.544954,0.080645,0.677419
3,Majority vote of discrete levels,0.512013,0.056452,0.661290


## Cut-Point Recalibration

The 40/80 boundaries in `normalize_label` are inherited from the scoring scale, not fitted to the
expert labels. Refitting them per model shows how much of each model's error is a scale offset
rather than a ranking failure. Cut points are fitted on training abstracts and scored on held-out
abstracts (grouped 5-fold, so no abstract appears on both sides), which keeps the reported gain
honest. The same protocol then tests whether dropping models from the ensemble actually helps.


In [96]:
CUT_POINT_GRID = np.arange(10, 121, 5)
CV_SPLITS = 5
CALIBRATED_AVERAGE_MODEL_NAME = "Average across models (cross-validated cuts)"


def fit_cut_points(scores: np.ndarray, y_true: np.ndarray) -> tuple[int, int]:
    candidates = [
        (quadratic_weighted_kappa(y_true, discretize(scores, lower, upper)), lower, upper)
        for lower in CUT_POINT_GRID
        for upper in CUT_POINT_GRID
        if upper > lower
    ]
    _, best_lower, best_upper = max(candidates)
    return int(best_lower), int(best_upper)


abstract_groups = raw_score_matrix.index.get_level_values(0).to_numpy()
expert_num_aligned = (
    average_model_df.set_index(pair_index).loc[raw_score_matrix.index, "expert_num"].to_numpy()
)
folds = list(GroupKFold(n_splits=CV_SPLITS).split(raw_score_matrix, expert_num_aligned, abstract_groups))

recalibration_rows = []
ensemble_held_out_pred = None
for model_name in raw_score_matrix.columns:
    scores = raw_score_matrix[model_name].to_numpy()
    held_out_pred = np.empty(len(scores), dtype=int)
    fold_cut_points = []
    for train_rows, test_rows in folds:
        lower_cut, upper_cut = fit_cut_points(scores[train_rows], expert_num_aligned[train_rows])
        fold_cut_points.append((lower_cut, upper_cut))
        held_out_pred[test_rows] = discretize(scores[test_rows], lower_cut, upper_cut)
    if model_name == AVERAGE_MODEL_NAME:
        ensemble_held_out_pred = held_out_pred
    recalibration_rows.append(
        {
            "Model": model_name,
            "QWK @ 40/80": quadratic_weighted_kappa(expert_num_aligned, discretize(scores, 40, 80)),
            "QWK @ refitted cuts": quadratic_weighted_kappa(expert_num_aligned, held_out_pred),
            "Mean signed error @ refitted cuts": float(np.mean(held_out_pred - expert_num_aligned)),
            "Exact accuracy @ refitted cuts": float(np.mean(held_out_pred == expert_num_aligned)),
            "Cut points (all data)": fit_cut_points(scores, expert_num_aligned),
            "Lower cut per fold": sorted({lower for lower, _ in fold_cut_points}),
            "Upper cut per fold": sorted({upper for _, upper in fold_cut_points}),
        }
    )

recalibration_df = pd.DataFrame(recalibration_rows)
recalibration_df["QWK gain"] = (
    recalibration_df["QWK @ refitted cuts"] - recalibration_df["QWK @ 40/80"]
)
recalibration_df = recalibration_df.sort_values("QWK @ refitted cuts", ascending=False).reset_index(
    drop=True
)
display(recalibration_df)

calibrated_ensemble_df = (
    average_model_df.set_index(pair_index).loc[raw_score_matrix.index].reset_index()
)
calibrated_ensemble_df["pred_num"] = ensemble_held_out_pred
fixed_class_metrics_df, fixed_overall_metrics_df = build_metrics_for_model(
    average_model_df, AVERAGE_MODEL_NAME
)
calibrated_class_metrics_df, calibrated_overall_metrics_df = build_metrics_for_model(
    calibrated_ensemble_df, CALIBRATED_AVERAGE_MODEL_NAME
)
calibration_overall_metrics_df = pd.concat(
    [fixed_overall_metrics_df, calibrated_overall_metrics_df], ignore_index=True
)
calibration_class_metrics_df = pd.concat(
    [fixed_class_metrics_df, calibrated_class_metrics_df], ignore_index=True
)
calibration_class_metrics_df["Class"] = pd.Categorical(
    calibration_class_metrics_df["Class"], categories=LABEL_ORDER, ordered=True
)

print("Average ensemble: fixed 40/80 vs cross-validated cut points")
display(calibration_overall_metrics_df)
display(
    calibration_class_metrics_df[
        ["Model", "Class", "Support", "Precision", "Recall", "One_vs_Rest_Accuracy"]
    ].sort_values(["Model", "Class"])
)

subset_scores = {
    subset: raw_score_matrix[list(subset)].mean(axis=1).to_numpy()
    for size in range(1, len(model_slugs) + 1)
    for subset in itertools.combinations(model_slugs, size)
}

subset_held_out_pred = np.empty(len(raw_score_matrix), dtype=int)
subset_picks = []
for train_rows, test_rows in folds:
    ranked = []
    for subset, scores in subset_scores.items():
        lower_cut, upper_cut = fit_cut_points(scores[train_rows], expert_num_aligned[train_rows])
        train_kappa = quadratic_weighted_kappa(
            expert_num_aligned[train_rows], discretize(scores[train_rows], lower_cut, upper_cut)
        )
        ranked.append((train_kappa, subset, lower_cut, upper_cut))
    _, subset, lower_cut, upper_cut = max(ranked, key=lambda row: row[0])
    subset_picks.append((subset, lower_cut, upper_cut))
    subset_held_out_pred[test_rows] = discretize(subset_scores[subset][test_rows], lower_cut, upper_cut)

all_models_kappa = recalibration_df.loc[
    recalibration_df["Model"] == AVERAGE_MODEL_NAME, "QWK @ refitted cuts"
].iloc[0]

print("Ensemble subset selection, scored on held-out abstracts:")
print(
    "  QWK with a subset chosen per fold: "
    f"{quadratic_weighted_kappa(expert_num_aligned, subset_held_out_pred):.3f}"
)
print(f"  QWK keeping all {len(model_slugs)} models:        {all_models_kappa:.3f}")
for subset, lower_cut, upper_cut in subset_picks:
    print(f"    fold picked {len(subset)} models at {lower_cut}/{upper_cut}: {', '.join(subset)}")

,Model,QWK @ 40/80,QWK @ refitted cuts,Mean signed error @ refitted cuts,Exact accuracy @ refitted cuts,Cut points (all data),Lower cut per fold,Upper cut per fold,QWK gain
0,Average across models,0.552577,0.514969,0.129032,0.629032,"(30, 80)","[30, 40]",[80],-0.037608
1,gemini-3.1-flash-lite,0.507937,0.449630,0.096774,0.540323,"(50, 80)","[40, 50, 75]",[80],-0.058306
2,gemma4-31b-cloud,0.421087,0.442159,-0.040323,0.588710,"(50, 80)",[50],"[80, 85, 90]",0.021073
3,mistral-large-3-675b-cloud,0.470817,0.434510,0.008065,0.604839,"(50, 85)","[35, 50]","[85, 100]",-0.036308
4,gpt-5.4-mini,0.512021,0.418577,-0.104839,0.588710,"(50, 80)","[20, 40, 50]","[80, 85, 90]",-0.093444


Average ensemble: fixed 40/80 vs cross-validated cut points


,Model,Quadratic Weighted Kappa,Mean signed error,Pairs,Unique abstracts
0,Average across models,0.562996,0.015385,130,21
1,Average across models (cross-validated cuts),0.514969,0.129032,124,20


,Model,Class,Support,Precision,Recall,One_vs_Rest_Accuracy
0,Average across models,Low,67,0.730159,0.686567,0.707692
1,Average across models,Moderate,49,0.563636,0.632653,0.676923
2,Average across models,High,14,0.750000,0.642857,0.938462
3,Average across models (cross-validated cuts),Low,64,0.744681,0.546875,0.669355
4,Average across models (cross-validated cuts),Moderate,47,0.523077,0.723404,0.645161
5,Average across models (cross-validated cuts),High,13,0.750000,0.692308,0.943548


Ensemble subset selection, scored on held-out abstracts:
  QWK with a subset chosen per fold: 0.474
  QWK keeping all 4 models:        0.515
    fold picked 2 models at 45/85: gpt-5.4-mini, mistral-large-3-675b-cloud
    fold picked 2 models at 45/85: gpt-5.4-mini, mistral-large-3-675b-cloud
    fold picked 3 models at 30/80: gemma4-31b-cloud, gpt-5.4-mini, mistral-large-3-675b-cloud
    fold picked 3 models at 30/85: gemma4-31b-cloud, gpt-5.4-mini, mistral-large-3-675b-cloud
    fold picked 3 models at 30/80: gemma4-31b-cloud, gpt-5.4-mini, mistral-large-3-675b-cloud


# Model Recommendation — Is the Cross-Model Average Better Than Isolated Models?

Cut points are methodologically defined (Low `< 40`, Moderate `40-80`, High `>= 80`) and are treated
as fixed throughout. Everything below is measured at those boundaries; no recalibration is proposed.

Basis: benchmark run `20260726_134607_4` — 5 models, scored independently against two expert sets,
`20260728_abstracts_evaluation_template_mark_om_v1` (Mark, 601 pairs / 91 abstracts) and
`20260728_abstracts_evaluation_template_charlie_s_v1` (Charlie, 720 pairs / 99 abstracts). The cells
above load Mark only; the Charlie column and the paired cluster bootstrap were produced by the same
logic run against the second expert set, and are reported here for robustness.

## Answer

**Yes against the average single model — no against the best one.** The cross-model average is never
significantly worse than any individual model against either expert, and against Mark it beats all
five. But it does not beat `deepseek-v4-flash-cloud`, which ties it on QWK against Charlie and leads
on accuracy and MAE against both experts.

Use the ensemble as the default. Its value is **insurance rather than a performance gain**: you do
not need to know in advance which model is strongest, and it neutralises `gemini-3.1-flash-lite`'s
failure mode (QWK 0.324 / 0.209) completely.

## Expert set A — Mark (601 pairs, 91 abstracts, expert mix 437/137/27)

| Model | QWK | Accuracy | MAE | Bias | Balanced acc. | Recall High |
|---|---|---|---|---|---|---|
| **ENSEMBLE (avg across models)** | **0.450** | 0.644 | 0.378 | +0.225 | **0.596** | **0.519** |
| claude-haiku-4.5 | 0.387 | 0.607 | 0.403 | +0.200 | 0.483 | 0.222 |
| deepseek-v4-flash-cloud | 0.383 | **0.682** | **0.354** | **+0.118** | 0.534 | 0.296 |
| mistral-small-2603 | 0.354 | 0.562 | 0.486 | +0.290 | 0.530 | 0.593 |
| gpt-5.4-mini | 0.354 | 0.637 | 0.421 | +0.151 | 0.467 | 0.333 |
| gemini-3.1-flash-lite | 0.324 | 0.444 | 0.606 | +0.532 | 0.536 | 0.667 |

Paired cluster bootstrap (resampling abstracts, 2000 draws, seed 0), ensemble QWK minus single-model
QWK — **all five intervals lie above zero**:

| vs. single model | Δ QWK | 95% interval | P(ensemble better) |
|---|---|---|---|
| gemini-3.1-flash-lite | +0.126 | [+0.075, +0.177] | 1.00 |
| gpt-5.4-mini | +0.096 | [+0.041, +0.153] | 1.00 |
| mistral-small-2603 | +0.095 | [+0.052, +0.141] | 1.00 |
| deepseek-v4-flash-cloud | +0.067 | [+0.016, +0.120] | 0.99 |
| claude-haiku-4.5 | +0.065 | [+0.008, +0.122] | 0.99 |

## Expert set B — Charlie (720 pairs, 99 abstracts, expert mix 616/104/0)

| Model | QWK | Accuracy | MAE | Bias | Balanced acc. | Recall Moderate |
|---|---|---|---|---|---|---|
| deepseek-v4-flash-cloud | **0.351** | **0.797** | **0.221** | **+0.096** | 0.638 | 0.413 |
| **ENSEMBLE (avg across models)** | 0.347 | 0.740 | 0.279 | +0.193 | 0.632 | 0.481 |
| claude-haiku-4.5 | 0.317 | 0.725 | 0.286 | +0.197 | **0.651** | **0.548** |
| gpt-5.4-mini | 0.306 | 0.757 | 0.275 | +0.156 | 0.590 | 0.356 |
| mistral-small-2603 | 0.275 | 0.692 | 0.353 | +0.256 | 0.548 | 0.346 |
| gemini-3.1-flash-lite | 0.209 | 0.528 | 0.525 | +0.489 | 0.532 | 0.538 |

| vs. single model | Δ QWK | 95% interval | Verdict |
|---|---|---|---|
| gemini-3.1-flash-lite | +0.138 | [+0.082, +0.196] | ensemble wins |
| mistral-small-2603 | +0.072 | [+0.026, +0.119] | ensemble wins |
| gpt-5.4-mini | +0.042 | [−0.017, +0.104] | not significant |
| claude-haiku-4.5 | +0.031 | [−0.017, +0.079] | not significant |
| deepseek-v4-flash-cloud | −0.003 | [−0.061, +0.061] | tied |

Charlie assigned **no** High labels, so High recall is undefined for every model in this column and
the ensemble's main advantage over `deepseek` in set A cannot be tested here.

## Why the answer depends on the metric

The ensemble leads on **QWK, balanced accuracy, and minority-class recall**. `deepseek-v4-flash-cloud`
leads on **plain accuracy, MAE, and bias** against both experts.

The mechanism is that the ensemble spreads its predictions across classes more, which recovers
minority-class cases — High recall 0.519 vs. `deepseek`'s 0.296 against Mark — and that is exactly
what QWK's chance correction and balanced accuracy reward. It pays for this with extra errors on the
dominant Low class, which is what plain accuracy and MAE penalise. Pair-for-pair, `deepseek` is
actually closer to the expert more often than the ensemble is:

| Ensemble vs. deepseek, per pair | Ensemble closer | Tie | Ensemble worse |
|---|---|---|---|
| Mark | 8.3% | 81.0% | 10.6% |
| Charlie | 3.6% | 86.9% | 9.4% |

So: if the deciding metric is ordinal agreement with the expert, keep the ensemble. If it is
exact-bucket accuracy or unbiasedness, `deepseek-v4-flash-cloud` alone is as good or better at one
fifth of the calls.

## Single-model rankings are unstable across experts

`mistral-small-2603` places 3rd against Mark and 5th against Charlie; `claude-haiku-4.5` 2nd and 3rd;
`gpt-5.4-mini` 4th and 4th. Only `deepseek-v4-flash-cloud` and the ensemble place 1st or 2nd against
both. This instability is the strongest practical argument for averaging: picking a single model on
one expert's labels risks picking a model that ranks mid-pack on another's.

## Caveats

- 91-99 abstracts per expert set, one annotator each. Single-model QWK differences of less than about
  0.05 are not resolvable at this sample size.
- The ensemble's clearest advantage — minority-class recall — rests on Mark's 27 High pairs, and
  Charlie's set has none.
- Every model over-predicts (bias +0.10 to +0.53 across both experts). With cut points fixed, this is
  a property to report, not one to tune away.
